In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path
import torch
import numpy as np

logging.basicConfig(level=logging.INFO)

sys.path.insert(0, str(Path.cwd().parents[0]))
sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

# Experiment Setup

In [ ]:
DATA_DIR = Path("../segm/fin_data/")

EXPERIMENT_NAME = "multi_run_loss_comparison"
RESULTS_DIR = Path("../segm/results") / EXPERIMENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 5
RUN_BASE_SEED = 12345

print(f"Dataset: {DATA_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Number of runs per loss: {N_RUNS}")

In [ ]:
MODELS_PATH = Path("../segm/models/")

TEST_ROOT = Path("../segm/data/test")
REAL_ROOT = Path("../segm/data/labeled_wo_test")
SYNTH_ROOT = Path("../segm/fin_data/clusters_wo_emboss")

# Data Loading

In [ ]:
from precipseg import data

LOAD_SYNTH_MODE = "basic"  # "basic" or "clustered"

# only used for "basic"
LOAD_NUM_SYNTH = 5  

# only used for "clustered"
CLUSTERS_BASE_COUNTS = (10, 13, 7, 9, 6)  
CLUSTERS_MULTIPLICATOR = 5
CLUSTERS_DIR_PREFIX = "cl"  # assumes the clustered images were generated like *{prefix}{i}/img.png

# test
test_img_paths, test_label_paths = data.load_paths(TEST_ROOT)
test_imgs = [data.read_image_normalized(p) for p in test_img_paths]
test_labels = [data.read_image_normalized(p) for p in test_label_paths]

# train real
real_img_paths, real_label_paths = data.load_paths(REAL_ROOT)

# train synthetic
if LOAD_SYNTH_MODE == "basic":
    synth_img_paths, synth_label_paths = data.load_paths(SYNTH_ROOT, LOAD_NUM_SYNTH)
elif LOAD_SYNTH_MODE == "clustered":
    synth_img_paths, synth_label_paths = data.load_clustered(SYNTH_ROOT, CLUSTERS_BASE_COUNTS, CLUSTERS_MULTIPLICATOR)
else:
    raise ValueError(f"Unknown LOAD_SYNTH_MODE={LOAD_SYNTH_MODE!r}")


train_img_paths = real_img_paths + synth_img_paths
train_label_paths = real_label_paths + synth_label_paths

print(f"Train: {len(real_img_paths)} real + {len(synth_img_paths)} synthetic ({LOAD_SYNTH_MODE}).")
print(f"Test: {len(test_img_paths)}.")

In [ ]:
from precipseg import data

PATCH_SIZE = 256
X_CHANNELS = 3
BATCH_SIZE = 16
VAL_SIZE = 0.25

train_dl, val_dl = data.build_dataloaders(
    train_img_paths, train_label_paths,
    patch_size=PATCH_SIZE,
    x_channels=X_CHANNELS,
    batch_size=BATCH_SIZE,
    val_size=VAL_SIZE,
)

In [ ]:
from precipseg.data import show_augmented_triple

show_augmented_triple(n_show=5, train_dl=train_dl)

# Loss Selection

In [ ]:
from precipseg.losses import available_losses
from itertools import product

print("Available losses:", available_losses())

LOSS_PARAM_GRID = {
    "focal": dict(
        alpha=[0.3, 0.5, 0.7],
        gamma=[1.5, 2.0, 2.5],
    ),
    "focal_dice": dict(
        alpha=[0.2, 0.4, 0.6],
        gamma=[1.5, 2.0, 2.5],
        smooth=[1e-6],
    ),
    "focal_tversky": dict(
        alpha=[0.3, 0.5, 0.7],
        beta=[0.7, 0.5, 0.3],
        gamma=[1.3],
        bce_weight=[0.3],
        smooth=[1e-6],
    ),
}


LOSSES = []

for loss_name, grid in LOSS_PARAM_GRID.items():
    for values in product(*grid.values()):
        LOSSES.append({"loss_name": loss_name, "params": dict(zip(grid.keys(), values))})

print(f"Number of loss configurations: {len(LOSS_CONFIGS)}")

# Training

In [ ]:
from precipseg.training import load_run_json, has_required_metrics, run_one

THRESHOLDS = np.linspace(0.10, 0.95, 10)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for cfg in LOSSES:
    loss_name = cfg["loss_name"]
    params = cfg["params"]

    tag = f"{loss_name}-" + "-".join(f"{k}{v}" for k, v in params.items())
    config_dir = RESULTS_DIR / tag
    config_dir.mkdir(parents=True, exist_ok=True)

    runs = []
    for i in range(N_RUNS):
        seed = RUN_BASE_SEED + i
        run_dir = config_dir / f"seed_{seed}"
        run_json = run_dir / "run.json"

        if run_json.exists():
            cached = load_run_json(run_json)
            if has_required_metrics(cached):
                runs.append(cached)
                continue

        runs.append(
            run_one(
                loss_name=loss_name,
                params=params,
                seed=seed,
                config_dir=config_dir,
                x_channels=X_CHANNELS,
                device=DEVICE,
                train_dl=train_dl,
                val_dl=val_dl,
                test_imgs=test_imgs,
                test_labels=test_labels,
                synth_root=str(SYNTH_ROOT),
                real_imgs=real_img_paths,
                synth_imgs=synth_img_paths,
                thresholds=THRESHOLDS,
                max_images=6,
            )
        )